In [1]:
import torch
import selfies as sf
from rdkit import Chem
from model_architecture_no_chemberta import Transformer,predict 
from rdkit.Chem import Draw
from rdkit import Chem
from rdkit.Chem import Crippen, QED
from rdkit.Chem import Descriptors
from rdkit.Chem import Lipinski
from rdkit.Chem.Scaffolds import MurckoScaffold

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
# cuda 있으면 cuda 쓰고 없으면 cpu 쓰기

# 체크포인트 로드
checkpoint = torch.load('D:\minimax\molecule_generation\\transformer_parameter\model_checkpoint_no_chemberta.pt', map_location=device, weights_only=False)
token2id = checkpoint['token2id']
id2token = checkpoint['id2token']

config = checkpoint['config'].copy()
config['max_len'] = checkpoint['max_len']

# 모델 생성
model = Transformer(**config) # 모델 파라미터 정보

# 학습된 파라미터 로드
model.load_state_dict(checkpoint['model_state_dict'], strict=False)

d:\minimax\.venv\lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


<All keys matched successfully>

In [3]:
import pubchempy as pcp
from rdkit.Chem import inchi

def find_molecule_exists(mol): # 새로 만들어진 물질이 기존에 존재하는 것인지 아닌지 확인
	inchikey = inchi.MolToInchiKey(mol)
	results = pcp.get_compounds(inchikey, "inchikey") # inchikey로 검색해서 결과가 존재하면 기존에 분자가 이미 있는 것
	if not results:
		return None

#### 리핀스키 5규칙 
- 우리가 먹는 약에는 공통적인 특성이 존재
	- (1)분자량은 500 달톤 이하 : Descriptors.MolWt(mol)
	- (2)logP <5 : Crippen.MolLogP(mol)
	- (3)수소 결합 주개가 5개 이하 
	- (4)수소 결합 받개가 10개 이하

- 이건 걍 내가 넣고 싶은 거
	- QED 계산 : 화합물이 약처럼 될 가능성을 수치화(0~1까지)
	- qed = QED.qed(mol)

In [4]:
def isit_available_medicine(mol):
	# logp 계산 : 분자가 수용성인지, 지용성인지
	logp = Crippen.MolLogP(mol)

	# QED 계산 : 화합물이 약처럼 될 가능성을 수치화(0~1까지)
	qed = QED.qed(mol)

	molecule_weight = Descriptors.MolWt(mol) # 500 이하

	hbd = Lipinski.NumHDonors(mol) # 수소 결합 주개(5개 이하)
	hba = Lipinski.NumHAcceptors(mol) # 수소 결합 받개(10개 이하)
	
	return [molecule_weight, logp, qed, hbd, hba]

In [5]:
def extract_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    cano_scaffold = Chem.MolToSmiles(scaffold, canonical=True)
    return cano_scaffold

In [6]:
def is_chemically_valid(smiles): # 분자의 화학적 유효성 검사
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False
    try:
        Chem.SanitizeMol(mol)
        Chem.MolToSmiles(mol, canonical=True)
    except:
        return False
    return True

In [11]:
lst = []

smiles = 'Cn1cnc2c1c(=O)n(c(=O)n2C)C'
cano_scaffold = extract_scaffold(smiles)
sca_list = list(sf.split_selfies(sf.encoder(cano_scaffold)))
token_sca_list = [token2id[i] for i in sca_list]

res = [token2id['[SOS]']] + token_sca_list + [token2id['[EOS]']] + [token2id['[PAD]']]*(checkpoint['max_len']-len(token_sca_list))

# Here we test some examples to observe how the model predicts
example = torch.tensor([res], dtype=torch.long, device=device)

def find_key_by_value(d, value):
    return [k for k, v in d.items() if v == value]

for i in range(1000):
	result = predict(model, example) 
	sf_string = ''.join([find_key_by_value(token2id, i)[0] for i in result[1:-1]])
	res_to_smiles,attr = sf.decoder(sf_string,attribute=True) # sf_string을 smiles로 바꾸기
	mol = Chem.MolFromSmiles(res_to_smiles) 
	if is_chemically_valid(res_to_smiles) and find_molecule_exists(mol) is None: # 만약 분자가 기존에 없는 것이라면
		medicine_standard = isit_available_medicine(mol) # 약이 될 수 있는지 관련 지표를 구해서 
		lst.append([f'new molecule{i}',res_to_smiles] + medicine_standard) # DB에 저장

d:\minimax\.venv\lib\site-packages\torch\nn\functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


In [13]:
import pandas as pd 

molecule_generation = pd.DataFrame(lst,columns=['molecule_name','new_smiles','molecule_weight', 'logP', 'QED', 'hbd', 'hba'])

In [14]:
molecule_generation.head()

,molecule_name,new_smiles,molecule_weight,logP,QED,hbd,hba
0,new molecule0,[C-1]#[N+1][N-1][PH1]([N-1][N+1]#N)[N-1][N+1]#N,156.049,2.43715,0.270668,0,2
1,new molecule1,[C-1]#[N+1][N-1][PH1]([N-1][N+1]#N)[N-1][N+1]#N,156.049,2.43715,0.270668,0,2
2,new molecule2,[C-1]#[N+1][N-1][PH1]([N-1][N+1]#N)[N-1][N+1]#N,156.049,2.43715,0.270668,0,2
3,new molecule3,[C-1]#[N+1][N-1][PH1]([N-1][N+1]#N)[N-1][N+1]#N,156.049,2.43715,0.270668,0,2
4,new molecule4,[C-1]#[N+1][N-1][PH1]([N-1][N+1]#N)[N-1][N+1]#N,156.049,2.43715,0.270668,0,2


In [15]:
molecule_generation.to_csv('transformer_predict_result_no_chemberta.csv',index=False)